In [ ]:
# youtube - https://www.youtube.com/watch?v=zCEJurLGFRk&t=14s
# % pip install google-api-python-client google-auth-httplib2 google-auth-oauthlib gspread
import gspread
from google.oauth2.service_account import Credentials
import pandas as pd
from gspread_dataframe import set_with_dataframe

scopes = [
    'https://www.googleapis.com/auth/spreadsheets']

creds = Credentials.from_service_account_file(
    'credential.json', scopes=scopes)

client = gspread.authorize(creds)

# 1PP6gpBcoOHjjgCx7LuHLa3dv4ET6ufKvpSY4UDvBczQ
sheet_id = '11Ora6_5EoQJdgnUpjjZgFZyrILguo1c32mde_uQwupw'
sheet = client.open_by_key(sheet_id)

df = pd.DataFrame({
    "Name": ["Alice", "Bob", "Charlie"],
    "Age": [25, 30, 40]
})

set_with_dataframe(sheet.sheet1, df) 

values_list = sheet.sheet1.get_all_values()
print(values_list)


# .worksheet(new_worksheet_name) -- to select worksheet page
# .add_worksheet(new_worksheet_name, rows=10, cols=10) -- add new worksheet page

[['Name', 'Age'], ['Alice', '25'], ['Bob', '30'], ['Charlie', '40']]


In [2]:
from data import Data

df = Data()
spi = df.sovereign_bond_yields()
spi.dtypes

Year        object
Region    category
Value      float64
dtype: object

In [3]:
ex_rate = df.forex_exchange()
ex_rate.dtypes

Year        object
Region    category
Value      float64
dtype: object

In [ ]:
values_list[1:]
# Convert the DataFrame back to a list of lists
values_to_update = df.values.tolist()

# Update the sheet with the new values
sheet.sheet1.update('A1', values_to_update)

[['hello', 'testing3'], ['', ''], ['helloi', '']]

In [15]:
df_with_header = [df.columns.tolist()] + df.values.tolist()
df_with_header

[['testing', 'testing 2 '], ['hello', 'testing3'], ['', ''], ['helloi', '']]

In [3]:
import pandas as pd 

df = pd.read_excel(r"C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Capital Flow Monitor\SEACEN CFM January 2025\edited files\Data\Section 1 Charts.xlsx",
            sheet_name='stock indices',
            header=2,)

df.head(20)

,Region,China,Hong Kong SAR (China),India,Indonesia,Malaysia,Philippines,Singapore,South Korea,Taiwan,Thailand,Unnamed: 11,Unnamed: 12,.T1,Dec-2006
0,Unit,19Dec1990=100,31Jul1964=100,1978-1979=100,10Aug1982=100,01Jan1977=100,28Feb1990=1022,09Jan2008=3344.53,04Jan1980=100,1966=100,30Apr1975=100,NaN,NaN,.TN,Oct-2023
1,Last Update Time,2023-10-31 00:00:00,2023-10-31 00:00:00,2023-10-31 00:00:00,2023-10-31 00:00:00,2023-10-31 00:00:00,2023-10-31 00:00:00,2023-10-31 00:00:00,2023-10-31 00:00:00,2023-11-07 00:00:00,2023-10-31 00:00:00,NaN,NaN,.SOURCE,MSCI
2,2010-01-01 00:00:00,2989.29,20121.99,16357.96,2610.796,1259.16,2953.19,2745.35,1602.43,8098.5795,696.55,NaN,NaN,Jan-2010,615.222
3,2010-02-01 00:00:00,3051.94,20608.7,16429.55,2549.033,1270.78,3043.75,2750.86,1594.58,7431.712143,721.37,NaN,NaN,Feb-2010,618.551
4,2010-03-01 00:00:00,3109.11,21239.35,17527.77,2777.301,1320.57,3161.8,2887.46,1692.85,7775.163913,787.98,NaN,NaN,Mar-2010,629.447
5,2010-04-01 00:00:00,2870.61,21108.59,17558.71,2971.252,1346.38,3290.09,2974.61,1741.56,8052.446667,763.51,NaN,NaN,Apr-2010,634.988
6,2010-05-01 00:00:00,2592.15,19765.19,16944.63,2796.957,1285.01,3272.73,2752.6,1641.25,7525.677143,750.43,NaN,NaN,May-2010,627.872
7,2010-06-01 00:00:00,2398.37,20128.99,17700.9,2913.684,1314.02,3372.71,2835.51,1698.29,7383.415714,797.31,NaN,NaN,Jun-2010,631.547
8,2010-07-01 00:00:00,2637.5,21029.81,17868.29,3069.28,1360.92,3426.95,2987.7,1759.33,7638.862273,855.83,NaN,NaN,Jul-2010,626.86
9,2010-08-01 00:00:00,2638.8,20536.49,17971.12,3081.884,1422.49,3566.23,2950.33,1742.75,7883.908636,913.19,NaN,NaN,Aug-2010,589.283


In [ ]:
import datetime
from pandasql import sqldf

df2 = df.drop([0,1])
df2.drop(columns=['Unnamed: 11', 'Unnamed: 12', '.T1'], inplace=True)
df2.rename(columns={'Dec-2006': 'Vietnam', 'Region': 'Date'}, inplace=True)
df2.reset_index(drop=True,inplace=True)
df2 = df2.iloc[:df2['Date'].isna().idxmax()]
df2 = df2.melt(id_vars="Date", var_name="Region", value_name="Value")
df2["Date"] = pd.to_datetime(df2["Date"]) + pd.offsets.MonthEnd(0)
df2["Year"] = df2["Date"].apply(lambda x: x.date().strftime("%Y"))
df2["Date2"] = df2["Date"].apply(lambda x: x.date().strftime("%Y-%m"))
year_now = datetime.date.today().year
year2 = [str(x) for x in range(2020, year_now + 1)]
country = df2["Region"].unique()
df2 = sqldf(
            f'select Date2 as Date, Year, Region, Value, row_number() over(partition by Region, Year order by Date2) as row_num from df2 where Year in ("{'", "'.join(year2)}")'
        )
df2 = df2.groupby("Year").apply(
            lambda x: x[x["row_num"].isin([x["row_num"].min(), x["row_num"].max()])]
        )
year2 = df2['Year'].unique() ## add this in fx
df2.reset_index(drop=True, inplace=True)
df2.fillna(0, inplace=True)
calc_country = dict()

for x in country:
    for y in year2:
        if len(df2[(df2["Region"] == x) & (df2["Year"] == str(y))]) == 1:
            continue
        df_temp = df2[
            (df2["Region"] == x)
            & (df2["Year"] == str(y))
        ]
        denomi = float(
            df_temp[(df_temp["row_num"] == df_temp["row_num"].min())
            ]["Value"].values[0]
        )
        nomi = float(
            df_temp[(df_temp["row_num"] == df_temp["row_num"].max())
            ]["Value"].values[0]
        )
        calc = 0 if nomi == 0 and denomi == 0 else ((nomi / denomi) - 1) * 100
        if calc_country.get(x) == None:
            calc_country.update({x: {y: calc}})
        else:
            calc_country[x].update({y: calc})

df3 = pd.DataFrame(calc_country).reset_index()
df3 = df3.melt(id_vars="index", var_name="Region", value_name="Value").rename(
    {"index": "Year"}, axis=1
)
region_sort = list(
    df3[(df3["Year"] == df3["Year"].unique()[-1])].sort_values(
        by="Value", ascending=False
    )["Region"]
)
df3["Region"] = pd.Categorical(
    df3["Region"], categories=region_sort, ordered=True
)
df3 = df3.sort_values(by=["Region", "Year"], ascending=False)
df3

C:\Users\AhmadAizudeen\AppData\Local\Temp\ipykernel_24020\3309472721.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df2 = df2.groupby("Year").apply(


,Year,Region,Value
39,2024,South Korea,-3.908550
38,2023,South Korea,9.492470
37,2022,South Korea,-16.030248
36,2021,South Korea,0.048384
35,2020,South Korea,35.604362
54,2024,Vietnam,-1.957916
53,2023,Vietnam,-1.024689
52,2022,Vietnam,-38.188689
51,2021,Vietnam,16.629364
50,2020,Vietnam,9.152209


In [4]:
import requests
import pandas as pd
from io import BytesIO

file_url = 'https://www.iif.com/Portals/0/Files/Databases/monthly_em_portfolio_flows_database.xlsx?ver=2025-03-13-063222-833'
response = requests.get(file_url)

if response.headers.get('Content-Type') == 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet':
    excel_data = BytesIO(response.content)
    df = pd.read_excel(excel_data, sheet_name='Country Flows_nw', header=3)
else:
    print("Failed to download a valid Excel file. The URL might not be a direct link.")

df

,Unnamed: 0,China,Unnamed: 2,India,Unnamed: 4,Indonesia,Unnamed: 6,Korea,Unnamed: 8,Malaysia,...,Lithuania,Unnamed: 55,Slovenia,Unnamed: 57,North Macedonia,Unnamed: 59,Mongolia,Unnamed: 61,Kenya,Ghana
0,NaT,Equity,Debt,Equity,Debt,Equity,Debt,Equity,Debt,Equity,...,Equity,Debt,Equity,Debt,Equity,Debt,Equity,Debt,Portfolio,Debt
1,2000-01-15,NaN,NaN,NaN,NaN,NaN,NaN,1416.7,-320.7,NaN,...,NaN,NaN,1.01545,-8.631325,NaN,NaN,NaN,NaN,NaN,NaN
2,2000-02-15,NaN,NaN,NaN,NaN,NaN,NaN,1716.8,662.9,NaN,...,NaN,NaN,-0.68936,-4.23464,NaN,NaN,NaN,NaN,NaN,NaN
3,2000-03-15,NaN,NaN,NaN,NaN,NaN,NaN,3665.3,1496.8,NaN,...,NaN,NaN,1.44693,357.295248,NaN,NaN,NaN,NaN,NaN,NaN
4,2000-04-15,NaN,NaN,NaN,NaN,NaN,NaN,235.5,673.9,NaN,...,NaN,NaN,1.796593,-1.040132,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
301,2025-01-15,2038.032575,4377.135097,-8418.06,-310.02,-229.283,133.23,-1002.16,-28.8,-701.7,...,-12.679661,2048.780421,-10.462792,1476.289592,NaN,NaN,0.25,24.08,NaN,NaN
302,2025-02-15,11430.544163,-15104.736471,-5353.17,-599.12,-1110.57,545.71,-2845.84,4077,-495.4,...,NaN,NaN,NaN,NaN,NaN,NaN,-0.32,112.22,NaN,NaN
303,2025-03-15,-8946.259283,-6672.359035,234.44,1046.84,-489.753,162.41,-1460.98,NaN,-1044.8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
304,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [97]:
df[df.columns[[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,60,61]]]
# [0,1,3,5,7,9,11,13,60]
# [0,2,4,6,8,10,12,61]
#0,1,2,3

,Unnamed: 0,China,Unnamed: 2,India,Unnamed: 4,Indonesia,Unnamed: 6,Korea,Unnamed: 8,Malaysia,Unnamed: 10,Philippines,Unnamed: 12,"Taiwan, China",Thailand,Unnamed: 15,Mongolia,Unnamed: 61
0,NaT,Equity,Debt,Equity,Debt,Equity,Debt,Equity,Debt,Equity,Debt,Equity,Debt,Equity,Equity,Debt,Equity,Debt
1,2000-01-15,NaN,NaN,NaN,NaN,NaN,NaN,1416.7,-320.7,NaN,NaN,NaN,NaN,1860.32,NaN,NaN,NaN,NaN
2,2000-02-15,NaN,NaN,NaN,NaN,NaN,NaN,1716.8,662.9,NaN,NaN,NaN,NaN,1779.85,NaN,NaN,NaN,NaN
3,2000-03-15,NaN,NaN,NaN,NaN,NaN,NaN,3665.3,1496.8,NaN,NaN,NaN,NaN,2071.92,NaN,NaN,NaN,NaN
4,2000-04-15,NaN,NaN,NaN,NaN,NaN,NaN,235.5,673.9,NaN,NaN,NaN,NaN,-218.81,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
301,2025-01-15,2038.032575,4377.135097,-8418.06,-310.02,-229.283,133.23,-1002.16,-28.8,-701.7,260.769231,-113.9786,NaN,-1261.47,-330.28,-358.41,0.25,24.08
302,2025-02-15,11430.544163,-15104.736471,-5353.17,-599.12,-1110.57,545.71,-2845.84,4077,-495.4,-251.752323,-145.0901,NaN,-3884.45,-195.18,145.6,-0.32,112.22
303,2025-03-15,-8946.259283,-6672.359035,234.44,1046.84,-489.753,162.41,-1460.98,NaN,-1044.8,NaN,49.7665,NaN,-13143.6,-646.6,618.04,NaN,NaN
304,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [113]:
equity = df[df.columns[[0,1,3,5,7,9,11,13,14,16,45,60]]]
equity.rename(columns={'Unnamed: 0': 'Date'}, inplace=True)
equity['Date'].fillna(0, inplace=True)
date_new = [x for x in equity['Date'] if isinstance(x, datetime.date)]
equity = equity[equity['Date'].isin(date_new)]
equity.set_index('Date', inplace=True)
equity

C:\Users\Admin\AppData\Local\Temp\ipykernel_25116\3513568812.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  equity.rename(columns={'Unnamed: 0': 'Date'}, inplace=True)
C:\Users\Admin\AppData\Local\Temp\ipykernel_25116\3513568812.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  equity['Date'].fillna(0, inplace=True)
C:\Users\Admin\AppDa

,China,India,Indonesia,Korea,Malaysia,Philippines,"Taiwan, China",Thailand,Vietnam,Sri lanka,Mongolia
Date,,,,,,,,,,,
2000-01-15,NaN,NaN,NaN,1416.7,NaN,NaN,1860.32,NaN,NaN,NaN,NaN
2000-02-15,NaN,NaN,NaN,1716.8,NaN,NaN,1779.85,NaN,NaN,NaN,NaN
2000-03-15,NaN,NaN,NaN,3665.3,NaN,NaN,2071.92,NaN,NaN,NaN,NaN
2000-04-15,NaN,NaN,NaN,235.5,NaN,NaN,-218.81,NaN,NaN,NaN,NaN
2000-05-15,NaN,NaN,NaN,786.5,NaN,NaN,322.09,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2024-11-15,-12703.761739,-2680.43,-1063.362,-3202.53,-698.5,-349.0686,-8045.04,-398.39,-468.4,-10.5,-1.66
2024-12-15,-1288.925091,1320.62,-312.638,-1530.1,-636.8,-103.1577,707.73,-293.94,-88.5,-1.7,-0.37
2025-01-15,2038.032575,-8418.06,-229.283,-1002.16,-701.7,-113.9786,-1261.47,-330.28,-254.81,-21.7,0.25


In [112]:
debt = df[df.columns[[0,2,4,6,8,10,12,15,61]]]
debt.rename(columns={'Unnamed: 0': 'Date'}, inplace=True)
debt['Date'].fillna(0, inplace=True)
date_new = [x for x in debt['Date'] if isinstance(x, datetime.date)]
debt = debt[debt['Date'].isin(date_new)]
debt.rename(columns={k:v for (k,v) in zip(debt.columns, equity.columns.drop(['Taiwan, China', 'Vietnam', 'Sri lanka']))}, inplace=True)
debt.set_index('Date', inplace=True)
debt.tail(15)

C:\Users\Admin\AppData\Local\Temp\ipykernel_25116\1794117280.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  debt.rename(columns={'Unnamed: 0': 'Date'}, inplace=True)
C:\Users\Admin\AppData\Local\Temp\ipykernel_25116\1794117280.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  debt['Date'].fillna(0, inplace=True)
C:\Users\Admin\AppData\L

,China,India,Indonesia,Korea,Malaysia,Philippines,Thailand,Mongolia
Date,,,,,,,,
2024-01-15,17590.033567,2534.81,-53.06,3880.1,-1081.006236,-360.1395,-111.7,182.12
2024-02-15,6195.362124,2373.87,-301.03,5065.3,-247.407096,169.4813,-428.85,-208.39
2024-03-15,4337.362325,2223.63,-1313.1,-4863.5,353.219837,1919.127,-255.37,24.77
2024-04-15,5766.33105,-1910.11,-1063.75,3571.3,121.887782,-644.4122,-697.45,4.22
2024-05-15,22141.955455,1005.05,1199.06,1253.1,1167.559524,2232.8947,510.72,198.56
2024-06-15,9546.422974,2136.15,117.42,-4564.8,-122.216336,37.9618,-287.43,9.27
2024-07-15,-3154.611912,2614.69,305.29,1944.9,1693.551889,1067.2096,779.31,-38.25
2024-08-15,3281.453951,1958.1,2492.13,4159.3,2085.737458,156.0395,761.55,84.58
2024-09-15,-34615.94187,153.02,1343.21,4132.8,253.925746,2476.0397,128.84,3.99


In [129]:
df_cf = pd.DataFrame(index=debt.index)

df_cf['Portfolio Debt'] = debt.sum(axis=1)
df_cf['Portfolio Equity'] = equity.sum(axis=1)
df_cf.reset_index(inplace=True)
df_cf

,Date,Portfolio Debt,Portfolio Equity
0,2000-01-15,-320.7,3277.02
1,2000-02-15,662.9,3496.65
2,2000-03-15,1496.8,5737.22
3,2000-04-15,673.9,16.69
4,2000-05-15,-1693.1,1108.59
...,...,...,...
298,2024-11-15,-45991.309863,-29621.642339
299,2024-12-15,-15178.846896,-2227.780791
300,2025-01-15,4097.984328,-10295.159025
301,2025-02-15,-11075.078795,-2983.445937


In [131]:
df_cf.melt(id_vars="Date", var_name="Portfolio", value_name="Value")

,Date,Portfolio,Value
0,2000-01-15,Portfolio Debt,-320.7
1,2000-02-15,Portfolio Debt,662.9
2,2000-03-15,Portfolio Debt,1496.8
3,2000-04-15,Portfolio Debt,673.9
4,2000-05-15,Portfolio Debt,-1693.1
...,...,...,...
601,2024-11-15,Portfolio Equity,-29621.642339
602,2024-12-15,Portfolio Equity,-2227.780791
603,2025-01-15,Portfolio Equity,-10295.159025
604,2025-02-15,Portfolio Equity,-2983.445937


In [51]:
import pandas as pd
from datetime import datetime
import os

df_init = pd.read_excel(r"C:\Users\Admin\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Desktop\Automation.xlsx", sheet_name='BoP edit')

path_dict = dict()
list_1 = ['IMF BOP Annual.xlsx', 'IMF BOP Quarterly.xlsx']
# directory_path= r'C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Roger and Aizudeen\Country BoP and IIP Data'
directory_path= r'C:\Users\Admin\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Roger and Aizudeen\Country BoP and IIP Data'
for filename in os.listdir(directory_path):
    if filename in ['01 Emerging Market', '02 G7 Countries']:
        continue
    file_path = os.path.join(directory_path, filename)
    list_temp = []
    for filename2 in os.listdir(file_path):
        if any(word in str(filename2) for word in list_1):            
            list_temp.append(filename2)
            path_dict[filename] = list_temp
        else:
            continue

df_path = []
for k, v in path_dict.items():
    df_path.append(os.path.join(directory_path, k, v[1]))
    print(v[1]) ## to check the file name

df_all_annual = pd.DataFrame()
for x, y in enumerate(df_path):
    df = pd.read_excel(y)
    df = df[df[df.columns[0]].isin(df_init['desc'])]
    df_all_annual = pd.concat([df_all_annual, df]) 

df_all_annual.rename(columns={df_all_annual.columns[0]: 'Type'}, inplace=True)

df_all_annual_2 = df_all_annual[df_all_annual.columns[:4].to_list() + [dt for dt in df_all_annual.columns[4:].sort_values(ascending=True).to_list() if dt >= datetime(2019, 1, 1)]]

def sg_row(df:pd.DataFrame, main:bool = True) -> pd.DataFrame:
    var = df.columns[:4] if main else df.columns[4:]
    return df[var]

sg_add = {
'FDI Assets':{'FDI Equity Assets': 0.74,'FDI Debt Assets': 0.26},
'FDI Liabilities': {'FDI Equity Liabilities': 0.85,'FDI Debt Liabilities': 0.15},
'Portfolio Assets': {'Portfolio Equity Assets': 0.57, 'Portfolio Debt Assets': 0.43}, 
'Portfolio Liabilities': {'Portfolio Equity Liabilities': 0.67, 'Portfolio Debt Liabilities': 0.33},
'Other Investment Assets': {'Currency and Deposits Assets': 0.5, 'Loans Assets': 0.29, 'Insurance and Pension Assets': 0.02, 'Trade Credits and Advances Assets': 0.09, 'OI Others Assets': 0.1},
'Other Investment Liabilities': {'Currency and Deposits Liabilities': 0.5, 'Loans Liabilities': 0.29, 'Insurance and Pension Liabilities': 0.02, 'Trade Credits and Advances Liabilities': 0.09, 'OI Others Liabilities': 0.1},
}

m = df_all_annual_2[(df_all_annual_2['Region']== 'Singapore')]['Type'].to_list()
df_init[(df_init['desc'].isin(m))]['group'].unique().tolist()

sg_item = [ 'BoP: Financial Account: Direct Investment: Assets',
            'BoP: Financial Account: Direct Investment: Liabilities',
            'BoP: Financial Account: Portfolio Investment: Assets',
            'BoP: Financial Account: Portfolio Investment: Liabilities',
            'BoP: Financial Account: Other Investment: Assets',
            'BoP: Financial Account: Other Investment: Liabilities']

sg_df = pd.DataFrame(columns=df_all_annual_2.columns)

# ensure the zip between sg_item and sg_add is correct
for x, y in zip(sg_item, sg_add.values()):
    multiplier = list(y.values())
    row_type = list(y.keys())
    df_temp = df_all_annual_2[(df_all_annual_2['Type'] == x) &(df_all_annual_2['Region']== 'Singapore')]
    # print(multiplier, row_type)
    for n in range(len(row_type)):
        col1 = sg_row(df_temp, main=True)
        col1.iat[0, 0] = list(df_init[(df_init['short_title']==row_type[n])]['desc'])[0]
        col2 = sg_row(df_temp, main=False) * multiplier[n]
        merged = pd.concat([col1, col2], axis=1)
        sg_df = pd.concat([sg_df, merged], axis=0)

df_init_tw = pd.read_excel(r"C:\Users\Admin\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Desktop\Automation.xlsx", sheet_name='Taiwan BoP')
df_tw = pd.read_excel(r"C:\Users\Admin\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Roger and Aizudeen\Country BoP and IIP Data\Chinese Taipei (Taiwan) (TW)\TW CB BOP Quarterly.xlsx")
df_tw.rename(columns={df_tw.columns[0]: 'Type'}, inplace=True)
df_tw = df_tw[df_tw['Type'].isin(df_init_tw['desc'])]

for x, y in enumerate(df_tw['Type'].to_list()):
    df_tw.iat[x,0] = list(df_init_tw[(df_init_tw['desc'] == y)]['desc2'])[0]

selected_columns = [col for col in df_tw.columns[4:] if pd.to_datetime(col) >= datetime(2019, 1, 1)]
final_columns = list(df_tw.columns[:4]) + selected_columns
df_tw = df_tw[final_columns]

# concat to the main df
df_all_annual_2 = pd.concat([df_all_annual_2, sg_df, df_tw], axis=0)

seacen_country = ['Papua New Guinea', 'Vietnam', 'Nepal', 'India', 'Indonesia', 'Laos', 'Sri Lanka', 'Hong Kong SAR (China)', 'Philippines', 'Taiwan', 'Malaysia', 'Mongolia', 'China', 'Cambodia', 'Thailand', 'Singapore', 'South Korea', 'Brunei', 'Myanmar']

for x in df_all_annual_2['Type'].values:
    var1 = list(set(df_all_annual_2[(df_all_annual_2['Type'] == x)]['Region'].values).symmetric_difference(set(seacen_country)))
    if len(var1) > 0:
        for y in var1:
           df_temp = pd.DataFrame({"Type": [x], "Region": [y]})
           df_all_annual_2 = pd.concat([df_all_annual_2, df_temp], axis=0)
    else:
        continue

df_all_annual_2.sort_values(by=['Type', 'Region'], inplace=True)
df_all_annual_2.reset_index(drop=True, inplace=True)

init_dict = df_init.to_dict()
df_dict = dict()
for x in init_dict['desc'].values():
    df_temp = df_all_annual_2[(df_all_annual_2[df_all_annual_2.columns[0]] == x)].reset_index(drop=True)
    df_dict[x] = df_temp

# calculate the sum of the columns (1H and 2H) for each year
num1, num2 = 0, 1
df_col = df_all_annual_2.columns[4:][:-1]
df_temp_col_1 = df_all_annual_2[df_all_annual_2.columns[:4]].reset_index(drop=True)

for x in range(len(df_col)//2):
    df_col_temp = (df_all_annual_2[df_col[num1]] + df_all_annual_2[df_col[num1 + 1]]).div(1000)
    year = df_col[num1].year
    df_col_temp.reset_index(drop=True, inplace=True)
    df_temp_col_1[f'{num2}H{year}'] = df_col_temp
    num1 += 2
    num2 += 1
    if num2 == 3:
        num2 = 1

df_temp_col_1.drop(columns=['Last Update Time', 'Unit'], inplace=True)

BN IMF BOP Quarterly.xlsx
KH IMF BOP Quarterly.xlsx
CH IMF BOP Quarterly.xlsx
HK IMF BOP Quarterly.xlsx
IN IMF BOP Quarterly.xlsx
ID IMF BOP Quarterly.xlsx
KR IMF BOP Quarterly.xlsx
LA IMF BOP Quarterly.xlsx
MY IMF BOP Quarterly.xlsx
MG IMF BOP Quarterly.xlsx
NP IMF BOP Quarterly.xlsx
PG IMF BOP Quarterly.xlsx
PH IMF BOP Quarterly.xlsx
SG IMF BOP Quarterly.xlsx
LK IMF BOP Quarterly.xlsx
TH IMF BOP Quarterly.xlsx
VN IMF BOP Quarterly.xlsx


C:\Users\Admin\AppData\Local\Temp\ipykernel_11952\1069088016.py:74: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  sg_df = pd.concat([sg_df, merged], axis=0)


In [208]:
calc_dict = {'Asia-Pacific': seacen_country,
             'Asian Advance Economies': ['Hong Kong SAR (China)', 'South Korea', 'Singapore', 'Taiwan'],
             'ASEAN5 Economies': ['Indonesia', 'Malaysia', 'Philippines', 'Thailand', 'Vietnam'],
             'Asian EDMEs': ['Brunei', 'Cambodia', 'Laos', 'Mongolia', 'Nepal', 'Sri Lanka', 'Papua New Guinea', 'Myanmar'],
             'China': ['China'],
             'India': ['India'],
             }

###################### this is for equity ######################

asset_row = df_init.iloc[[0,4,5,6, 8, 9,10,11,12,13,14]]['desc'].to_list()
asset_row2 = ['Direct Investment', 'Portfolio Equity', 'Portfolio Debt','Financial Derivatives', 'Other Equity','Currency & Deposits','Loans','Insurance, Pension & Standardized Guarantee Schemes'
              ,'Trade Credit & Advances','Other Accounts Receivable','Reserve Assets', ]
sum_rows = df_init[(df_init['short_title'].isin(['OI Equity Assets','Insurance and Pension Assets','OI Others Assets',]))]['desc'].to_list()
order = ['Direct Investment','Portfolio Equity','Portfolio Debt','Financial Derivatives','Currency & Deposits','Loans','Trade Credit & Advances','Other Investments','Reserve Assets',]
df_by_region_main = pd.DataFrame(columns=df_temp_col_1.columns)

for x,y in calc_dict.items():
    # print(x,y)
    df_by_region_temp = df_temp_col_1[(df_temp_col_1['Region'].isin(y)) & (df_temp_col_1['Type'].isin(asset_row))]
    df_by_region_temp = df_by_region_temp.groupby(['Type']).sum()
    df_by_region_temp.loc[str(x) + ' Total'] = df_by_region_temp.sum()
    df_by_region_temp.loc['Other Investments'] = df_by_region_temp.loc[sum_rows].sum()
    df_by_region_temp.reset_index(inplace = True)
    df_by_region_temp = df_by_region_temp[(~df_by_region_temp['Type'].isin(sum_rows))] #####  the symbol of '~' use similar like notin, the oposite of isin. this is build in python operator
    df_by_region_temp['Type'] = df_by_region_temp['Type'].replace(asset_row, asset_row2)
    df_by_region_temp = df_by_region_temp.sort_values(by='Type', key=lambda x: x.map({k:i for i, k in enumerate(order)}))
    df_by_region_temp['Region'] = x

    df_by_region_main = pd.concat([df_by_region_main, df_by_region_temp], axis = 0)  


df_by_region_main.reset_index(inplace = True, drop=True)
df_by_region_main.insert(loc=0, column='Group', value='Asset')

###################### this is for equity ######################

C:\Users\Admin\AppData\Local\Temp\ipykernel_11952\1436010081.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_by_region_main = pd.concat([df_by_region_main, df_by_region_temp], axis = 0)


In [209]:
###################### this is for debt ######################

debt_row = df_init.iloc[[20,24,25,26,28,29,30,31,32,33,34]]['desc'].to_list()
debt_row2 = ['Direct Investment', 'Portfolio Equity', 'Portfolio Debt','Financial Derivatives', 'Other Equity','Currency & Deposits','Loans','Insurance, Pension & Standardized Guarantee Schemes'
              ,'Trade Credit & Advances','Other Accounts Payable','SDR Liabilities', ]
sum_rows2 = df_init[(df_init['short_title'].isin(['OI Equity Liabilities', 'Insurance and Pension Liabilities', 'OI Others Liabilities', 'SDR Liabilities']))]['desc'].to_list()
order2 = ['Direct Investment','Portfolio Equity','Portfolio Debt','Financial Derivatives','Currency & Deposits','Loans','Trade Credit & Advances','Other Investments']
df_by_region_main2 = pd.DataFrame(columns=df_temp_col_1.columns)

for x,y in calc_dict.items():
    # print(x,y)
    df_by_region_temp = df_temp_col_1[(df_temp_col_1['Region'].isin(y)) & (df_temp_col_1['Type'].isin(debt_row))]
    df_by_region_temp = df_by_region_temp.groupby(['Type']).sum()
    df_by_region_temp.loc[str(x) + ' Total'] = df_by_region_temp.sum()
    df_by_region_temp.loc['Other Investments'] = df_by_region_temp.loc[sum_rows2].sum()
    df_by_region_temp.reset_index(inplace = True)
    df_by_region_temp = df_by_region_temp[(~df_by_region_temp['Type'].isin(sum_rows2))] #####  the symbol of '~' use similar like notin, the oposite of isin. this is build in python operator
    df_by_region_temp['Type'] = df_by_region_temp['Type'].replace(debt_row, debt_row2)
    df_by_region_temp = df_by_region_temp.sort_values(by='Type', key=lambda x: x.map({k:i for i, k in enumerate(order2)}))
    df_by_region_temp['Region'] = x

    df_by_region_main2 = pd.concat([df_by_region_main2, df_by_region_temp], axis = 0)  

df_by_region_main2.reset_index(inplace = True, drop=True)
df_by_region_main2.insert(loc=0, column='Group', value='Debt')

###################### this is for debt ######################

C:\Users\Admin\AppData\Local\Temp\ipykernel_11952\92705744.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_by_region_main2 = pd.concat([df_by_region_main2, df_by_region_temp], axis = 0)


In [152]:
region_total = [str(x) + ' Total' for x in list(calc_dict.keys())]
debt_region_total = df_by_region_main2[(df_by_region_main2['Type'].isin(region_total))]
equity_region_total = df_by_region_main[(df_by_region_main['Type'].isin(region_total))]
equity_region_total

,Type,Region,1H2019,2H2019,1H2020,2H2020,1H2021,2H2021,1H2022,2H2022,1H2023,2H2023,1H2024,2H2024,1H2025
9,Asia-Pacific Total,Asia-Pacific,389.044691,547.351999,628.510252,978.900994,1065.303164,916.619511,559.260730,344.489349,442.993304,552.678233,706.164290,0.0,0.0
19,Asian Advance Economies Total,Asian Advance Economies,134.509632,260.204636,263.755677,344.696333,400.667114,398.601717,329.087641,88.959942,195.697465,309.143156,342.194245,0.0,0.0
29,ASEAN5 Economies Total,ASEAN5 Economies,56.164779,63.717382,48.154772,69.473275,46.370944,74.244961,19.548402,34.111901,19.843826,48.074688,28.277899,0.0,0.0
39,Asian EDMEs Total,Asian EDMEs,3.747470,3.693605,4.619171,4.636569,-4.726849,-0.950685,0.118859,5.081300,7.191929,8.329258,4.616460,0.0,0.0
49,China Total,China,115.034980,144.005664,220.706407,454.478225,536.591354,358.313976,158.824513,176.295051,127.195550,100.996875,214.436319,0.0,0.0
59,India Total,India,79.587829,75.730713,91.274225,105.616592,86.400601,86.409543,51.681316,40.041155,93.064534,86.134256,116.639367,0.0,0.0


In [210]:
###################### this is for caq data ######################

caq_data = df_temp_col_1[(df_temp_col_1['Type'].isin(df_init[(df_init['group'] == 'Current Account')]['desc'].to_list()[1:]))]
caq_data_calc = caq_data.groupby(['Type']).sum()
caq_data_calc.loc['Current Account Balance'] = caq_data_calc.sum()
caq_data_calc['Region'] = 'Asia-Pacific'
caq_data_calc.reset_index(inplace=True)
rename_to = ['Goods Trade', 'Services Trade','Primary Income','Secondary Income',]
rename_from = ['BoP: Current Account: Goods','BoP: Current Account: Services','BoP: Current Account: Primary Income','BoP: Current Account: Secondary Income',]
caq_data_calc['Type'] = caq_data_calc['Type'].replace(rename_from, rename_to)
caq_data_calc = caq_data_calc.sort_values(by='Type', key=lambda x: x.map({k:i for i, k in enumerate(rename_to)}))
caq_data_calc.reset_index(drop=True, inplace=True)
caq_data_calc.insert(loc=0, column='Group', value='Current Account')

###################### this is for caq data ######################

In [212]:
df_main = pd.concat([df_by_region_main, df_by_region_main2, caq_data_calc], axis = 0)
df_main.reset_index(drop=True, inplace=True)

In [213]:
df_main

,Group,Type,Region,1H2019,2H2019,1H2020,2H2020,1H2021,2H2021,1H2022,2H2022,1H2023,2H2023,1H2024,2H2024,1H2025
0,Asset,Direct Investment,Asia-Pacific,149.428150,180.510555,180.620470,185.833686,225.411431,243.147163,273.815687,238.898066,224.552799,231.259167,241.250903,0.0,0.0
1,Asset,Portfolio Equity,Asia-Pacific,69.708763,90.828745,90.133885,203.799634,220.565885,114.627352,100.883862,79.488389,83.948592,96.053443,184.612859,0.0,0.0
2,Asset,Portfolio Debt,Asia-Pacific,141.731881,66.638945,53.630082,35.342125,75.610734,81.595491,128.051328,122.462622,125.556845,65.334081,193.687866,0.0,0.0
3,Asset,Financial Derivatives,Asia-Pacific,-30.791857,-19.369328,-31.317723,-28.782652,-58.157658,-33.886654,-26.049555,-67.262978,-17.171229,-53.294229,-32.694167,0.0,0.0
4,Asset,Currency & Deposits,Asia-Pacific,3.800398,159.859281,115.000415,181.996993,163.467292,166.645973,129.161447,61.908016,-50.352414,158.941185,17.397668,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114,Current Account,Goods Trade,Asia-Pacific,178.355730,287.742185,242.101627,517.215196,316.757596,415.787977,342.325617,348.479502,305.626122,379.369215,350.592047,0.0,0.0
115,Current Account,Services Trade,Asia-Pacific,-70.253046,-74.170751,-48.635109,-43.736064,-14.060979,8.442623,30.986046,21.566123,-22.323167,-37.220262,-38.566699,0.0,0.0
116,Current Account,Primary Income,Asia-Pacific,-50.062974,-94.529809,-83.737764,-121.128304,-99.273719,-141.104465,-186.079393,-124.988884,-124.714608,-146.901440,-136.911042,0.0,0.0
117,Current Account,Secondary Income,Asia-Pacific,62.674586,67.011694,59.721253,74.581517,70.312601,76.773974,77.121667,90.834123,83.470313,92.437762,87.964756,0.0,0.0
